In [0]:
%sql

-- ============================================================
-- 1. BRONZE / SILVER / QUARANTINE RECONCILIATION
-- ============================================================

SELECT

    (
        SELECT COUNT(*)
        FROM clinical_trial_intelligence.bronze.edc_visits
    ) AS bronze_rows,

    (
        SELECT COUNT(*)
        FROM clinical_trial_intelligence.quarantine.visits
    ) AS quarantine_rows,

    (
        SELECT COUNT(*)
        FROM clinical_trial_intelligence.silver.visits
    ) AS silver_rows,

    (
        SELECT COUNT(DISTINCT visit_id)
        FROM clinical_trial_intelligence.silver.visits
    ) AS distinct_silver_visits;

In [0]:
%sql

-- ============================================================
-- 2. QUARANTINE VALIDATION
-- Expected quarantine rows from exploration: 320
-- ============================================================

SELECT
    COUNT(*) AS quarantine_rows,

    COUNT_IF(
        dq_failure_reasons IS NULL
        OR TRIM(dq_failure_reasons) = ''
    ) AS missing_failure_reason_rows,

    COUNT(DISTINCT _source_file_name)
        AS affected_source_files

FROM clinical_trial_intelligence.quarantine.visits;

In [0]:
%sql

-- ============================================================
-- 3. QUARANTINE DQ FAILURE DISTRIBUTION
-- ============================================================

SELECT
    failure_reason,
    COUNT(*) AS failed_record_count

FROM (
    SELECT
        EXPLODE(_dq_failures) AS failure_reason
    FROM clinical_trial_intelligence.quarantine.visits
)

GROUP BY failure_reason

ORDER BY
    failed_record_count DESC,
    failure_reason;

In [0]:
%sql

-- ============================================================
-- 4. VISIT BUSINESS-KEY INTEGRITY
--
-- Expected:
--   duplicate_visit_ids = 0
--   invalid_visit_ids   = 0
-- ============================================================

SELECT

    COUNT(*) - COUNT(DISTINCT visit_id)
        AS duplicate_visit_ids,

    COUNT_IF(
        visit_id IS NULL
        OR TRIM(visit_id) = ''
    ) AS invalid_visit_ids

FROM clinical_trial_intelligence.silver.visits;

In [0]:
%sql

-- ============================================================
-- 5. SILVER VISIT BUSINESS / CLINICAL INTEGRITY
-- ============================================================

SELECT

    COUNT(*) AS total_silver_rows,

    COUNT_IF(
        visit_id IS NULL
        OR TRIM(visit_id) = ''
    ) AS invalid_visit_ids,

    COUNT_IF(
        subject_id IS NULL
        OR TRIM(subject_id) = ''
    ) AS invalid_subject_ids,

    COUNT_IF(
        study_id IS NULL
        OR TRIM(study_id) = ''
    ) AS invalid_study_ids,

    COUNT_IF(
        site_id IS NULL
        OR TRIM(site_id) = ''
    ) AS invalid_site_ids,

    COUNT_IF(visit_date IS NULL)
        AS missing_visit_dates,

    COUNT_IF(
        visit_status NOT IN (
            'COMPLETED',
            'MISSED',
            'RESCHEDULED',
            'UNKNOWN'
        )
        OR visit_status IS NULL
    ) AS invalid_status_rows

FROM clinical_trial_intelligence.silver.visits;

In [0]:
%sql

-- ============================================================
-- 6. SILVER VISIT REFERENTIAL INTEGRITY
-- Expected all orphan counts = 0
-- ============================================================

SELECT
    'VISIT_TO_SUBJECT' AS validation_rule,
    COUNT(*) AS failed_rows

FROM clinical_trial_intelligence.silver.visits v

LEFT JOIN (
    SELECT DISTINCT subject_id
    FROM clinical_trial_intelligence.silver.subjects
) s
    ON v.subject_id = s.subject_id

WHERE s.subject_id IS NULL

UNION ALL

SELECT
    'VISIT_TO_STUDY',
    COUNT(*)

FROM clinical_trial_intelligence.silver.visits v

LEFT JOIN clinical_trial_intelligence.silver.dim_study d
    ON v.study_id = d.study_id

WHERE d.study_id IS NULL

UNION ALL

SELECT
    'VISIT_TO_SITE',
    COUNT(*)

FROM clinical_trial_intelligence.silver.visits v

LEFT JOIN clinical_trial_intelligence.silver.dim_site d
    ON v.site_id = d.site_id

WHERE d.site_id IS NULL;

In [0]:
%sql

-- ============================================================
-- 7. KNOWN VISIT RECORD VERIFICATION
--
-- Purpose:
-- Verify that Silver correctly represents visit summaries for
-- subjects with the most visit records.
-- Since visits are append-only (no SCD Type 2), this confirms
-- all expected visit rows were streamed through and landed
-- with the correct status distribution.
-- ============================================================

SELECT
    subject_id,
    COUNT(*)                               AS visit_count,
    MIN(visit_date)                        AS first_visit_date,
    MAX(visit_date)                        AS last_visit_date,
    COUNT_IF(visit_status = 'COMPLETED')   AS completed_visits,
    COUNT_IF(visit_status = 'MISSED')      AS missed_visits,
    COUNT_IF(visit_status = 'RESCHEDULED') AS rescheduled_visits,
    COUNT_IF(visit_status = 'UNKNOWN')     AS unknown_visits

FROM clinical_trial_intelligence.silver.visits

WHERE subject_id IN (

    SELECT subject_id
    FROM clinical_trial_intelligence.silver.visits
    GROUP BY subject_id
    ORDER BY COUNT(*) DESC
    LIMIT 3

)

GROUP BY subject_id

ORDER BY visit_count DESC, subject_id;

In [0]:
%sql

-- ============================================================
-- 8. FINAL VISIT PIPELINE VALIDATION GATE
--
-- Expected result: ZERO ROWS
-- ============================================================

SELECT
    'DUPLICATE_VISIT_ID' AS failed_check,
    COUNT(*) - COUNT(DISTINCT visit_id) AS failed_rows

FROM clinical_trial_intelligence.silver.visits

HAVING COUNT(*) - COUNT(DISTINCT visit_id) > 0

UNION ALL

SELECT
    'INVALID_VISIT_KEY',
    COUNT(*)

FROM clinical_trial_intelligence.silver.visits

WHERE visit_id IS NULL
   OR TRIM(visit_id) = ''

HAVING COUNT(*) > 0

UNION ALL

SELECT
    'MISSING_VISIT_DATE',
    COUNT(*)

FROM clinical_trial_intelligence.silver.visits

WHERE visit_date IS NULL

HAVING COUNT(*) > 0

UNION ALL

SELECT
    'INVALID_VISIT_STATUS',
    COUNT(*)

FROM clinical_trial_intelligence.silver.visits

WHERE visit_status NOT IN (
    'COMPLETED',
    'MISSED',
    'RESCHEDULED',
    'UNKNOWN'
)

HAVING COUNT(*) > 0;

# VISIT SILVER VALIDATION — FINAL RESULT

Status: PASS

Validation confirmed:

- Bronze visit records: 18,262
- Quarantined records: 320
- Silver rows: 17,942
- Distinct Silver visits: 17,942
- No duplicate visit identifiers
- No invalid Silver business keys
- DQ failure reasons retained for all quarantined records
- Visit status values standardized to expected domain (COMPLETED / MISSED / RESCHEDULED / UNKNOWN)
- All Silver visit records reference valid subjects, studies, and sites
- Final visit validation gate returned zero failures

Conclusion:
The Silver visit pipeline successfully separates invalid source
records into quarantine while maintaining validated visit data as
an append-only streaming table. Business-key integrity, DQ
traceability, visit status standardization, and referential
integrity were successfully validated.

VISIT SILVER PROCESSING: COMPLETE